In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
from pprint import pprint
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

import numpy as np
from pydantic_yaml import parse_yaml_file_as
import torch
from transformers import AutoTokenizer, PreTrainedTokenizer

from mllm.exp.args import MIXED_DECODER_MODEL_CFG_FNAME
from mllm.model.mixed_decoder import MixedDecoder
from mllm.config.model import MixedDecoderCfg
from mllm.data.qna.batch import QnaBatch
from mllm.data.qna.dataset import QnaDatasetAgg, QnaDatasetType, QNA_DATASETS_DEFAULT, QnaBaseDataset, load_qna_datasets

## Config

In [ ]:
DATA_PATH = Path('Q:/data')

TRAIN_ENCDEC_BERT_PATH = DATA_PATH / 'train_mllm_encdec_bert'
mixed_decoder_subdir = 'mixeddecoder-20260308_222546-pre_mixeddecoder20260308113953-bertbaseuncased-d768-embEncCls-inp128-decGpt2-decmgpt2-msl384-sepT-pallF-eer4-ewn10x10-frzencF-dsQna-msk_sep0.5x0.15_seq0.5x0.2x20_last0-trn_lr2.5e-05_bs10'

mixed_decoder_train_path = TRAIN_ENCDEC_BERT_PATH / mixed_decoder_subdir
mixed_decoder_snapshot_fpath = mixed_decoder_train_path / 'best.pth'
mixed_decoder_model_cfg_fpath = mixed_decoder_train_path / MIXED_DECODER_MODEL_CFG_FNAME

device_name = 'cpu'
# device_name = 'cuda'

device = torch.device(device_name)
print(device)

## Model loading

In [ ]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
pprint(model_cfg.dict())

tkz = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
inp_len = model_cfg.enc_bert.inp_len
model_cfg.emb_win_min_size = model_cfg.emb_win_max_size

chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None

## Load QnA datasets (train + val)

In [ ]:
max_chunks = max(model_cfg.emb_win_max_size, 1)
qna_agg_train, qna_agg_val = load_qna_datasets(
    tkz=tkz, inp_len=inp_len, max_chunks=max_chunks,
    cache_dir=str(DATA_PATH),
)
print(f'Train size: {len(qna_agg_train)}. Val size: {len(qna_agg_val)}.')
print(f'Train datasets: {len(qna_agg_train.datasets)}. Val datasets: {len(qna_agg_val.datasets)}.')
for i, ds in enumerate(qna_agg_train.datasets):
    print(f'  Train ds {i}: {type(ds).__name__} — {len(ds)} items')
for i, ds in enumerate(qna_agg_val.datasets):
    print(f'  Val ds {i}: {type(ds).__name__} — {len(ds)} items')

## Load test splits (where available)

In [ ]:
from datasets import load_dataset as hf_load_dataset
from mllm.data.qna.ds_03_triviaqa import TriviaQADataset, TRIVIAQA_HF_ID, TRIVIAQA_SUBSET\n
from mllm.data.qna.ds_07_mrqa import MrqaDataset, MRQA_HF_ID
from mllm.data.qna.ds_08_adversarialqa import AdversarialqaDataset, ADVERSARIALQA_HF_ID, ADVERSARIALQA_SUBSET\n

ds_kwargs = dict(tkz=tkz, inp_len=inp_len, max_chunks=max_chunks, device=device)

test_datasets: List[QnaBaseDataset] = []

# TriviaQA test split
ds_triviaqa_test_hf = hf_load_dataset(TRIVIAQA_HF_ID, TRIVIAQA_SUBSET, split='test', cache_dir=str(DATA_PATH))\n
test_datasets.append(TriviaQADataset(ds=ds_triviaqa_test_hf, **ds_kwargs))
print(f'TriviaQA test: {len(test_datasets[-1])} items')

# MRQA test split
ds_mrqa_test_hf = hf_load_dataset(MRQA_HF_ID, split='test', cache_dir=str(DATA_PATH))
test_datasets.append(MrqaDataset(ds=ds_mrqa_test_hf, **ds_kwargs))
print(f'MRQA test: {len(test_datasets[-1])} items')

# AdversarialQA test split
ds_advqa_test_hf = hf_load_dataset(ADVERSARIALQA_HF_ID, ADVERSARIALQA_SUBSET, split='test', cache_dir=str(DATA_PATH))\n
test_datasets.append(AdversarialqaDataset(ds=ds_advqa_test_hf, **ds_kwargs))
print(f'AdversarialQA test: {len(test_datasets[-1])} items')

qna_agg_test = QnaDatasetAgg(test_datasets, device=device)
print(f'\nTest aggregate size: {len(qna_agg_test)}')

## Autoregressive QnA generation function

In [ ]:
@torch.no_grad()
def generate_qna(
    model: MixedDecoder, qna_batch: QnaBatch,
    max_new_tokens: int = 100, temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive generation for QnA: encode context chunks, build per-sample
    context embedding windows, then generate answer tokens one by one.

    Args:
        model: MixedDecoder model in eval mode
        qna_batch: QnaBatch with context chunks, prompts, and answer tokens
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (<=0 means greedy argmax)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    cfg = model.cfg
    batch_size = len(qna_batch.ctx_chunk_counts)
    device = qna_batch.ctx_chunks_toks.device

    # 1. Encode all context chunks
    all_enc_embs = model.run_enc(qna_batch.ctx_chunks_toks, qna_batch.ctx_chunks_att_mask)

    # 2. Build per-sample context windows (own chunks + zero-padding if needed)
    emb_win_max = max(cfg.emb_win_max_size, 1)
    target_win_size = emb_win_max

    chunk_offsets = [0]
    for c in qna_batch.ctx_chunk_counts:
        chunk_offsets.append(chunk_offsets[-1] + c)

    ctx_embs_raw = torch.zeros((batch_size, target_win_size, cfg.d_model), device=device)
    for i in range(batch_size):
        start, end = chunk_offsets[i], chunk_offsets[i + 1]
        own = all_enc_embs[start:end]
        n_own = min(own.shape[0], target_win_size)
        ctx_embs_raw[i, :n_own] = own[:n_own]

    # 3. Apply emb_exp expansion or projection
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(ctx_embs_raw)
        exp_embs = exp_embs.view(batch_size, target_win_size * cfg.emb_exp_rate, model.d_dec)
        ctx_embs = exp_embs
    else:
        ctx_embs = ctx_embs_raw

    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]
    sep_len = 1 if cfg.use_sep else 0

    # 4. Build per-sample initial prefix: [ctx_embs, (sep), prompt_embs]
    prompt_lens = qna_batch.prompt_lengths
    max_prefix_len = n_ctx + sep_len + max(prompt_lens)

    input_embs = torch.zeros((batch_size, max_prefix_len, model.d_dec), device=device)
    attention_mask = torch.zeros((batch_size, max_prefix_len), dtype=torch.long, device=device)

    prompt_embs_all = model.word_embeddings(qna_batch.prompt_toks)
    if cfg.use_sep:
        sep_emb = model.word_embeddings(
            torch.full((1, 1), model.sep_token_id, dtype=torch.long, device=device)
        )

    for i in range(batch_size):
        pos = 0
        input_embs[i, :n_ctx] = ctx_embs[i]
        attention_mask[i, :n_ctx] = 1
        pos = n_ctx

        if cfg.use_sep:
            input_embs[i, pos:pos + 1] = sep_emb[0]
            attention_mask[i, pos:pos + 1] = 1
            pos += 1

        pl = prompt_lens[i]
        input_embs[i, pos:pos + pl] = prompt_embs_all[i, :pl]
        attention_mask[i, pos:pos + pl] = 1

    # 5. Autoregressive generation
    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
        pos_embs = model.pos_emb(pos_ids)
        embs_with_pos = input_embs + pos_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        next_emb = model.word_embeddings(next_tok.unsqueeze(1))
        input_embs = torch.cat([input_embs, next_emb], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)

    return torch.stack(generated_ids, dim=1)

## Helper: run inference on a split

In [ ]:
def run_qna_inference(
    model: MixedDecoder, tkz: PreTrainedTokenizer,
    ds_agg: QnaDatasetAgg, split_name: str,
    batch_off: int = 0, batch_size: int = 5,
    max_new_tokens: int = 60,
):
    """Run teacher-forced forward pass and autoregressive generation on a QnaDatasetAgg split."""
    cfg = model.cfg
    inds = list(range(batch_off, batch_off + batch_size))
    batch = ds_agg.get_batch(inds)

    # --- Teacher-forced forward pass ---
    with torch.no_grad():
        loss_dict, logits = model.run_on_qna(batch)
    print(f'=== {split_name} — teacher-forced (batch_off={batch_off}, batch_size={batch_size}) ===')
    print(f'Loss: {loss_dict}')
    print(f'Logits shape: {logits.shape}')
    print()

    emb_win_max = max(cfg.emb_win_max_size, 1)
    n_ctx = emb_win_max
    if cfg.emb_exp_rate > 0:
        n_ctx = n_ctx * cfg.emb_exp_rate
    sep_len = 1 if cfg.use_sep else 0

    chunk_offset = 0
    for i in range(batch_size):
        n_chunks = batch.ctx_chunk_counts[i]
        pl = batch.prompt_lengths[i]
        al = int(batch.ans_att_mask[i].sum().item())
        target_start = n_ctx + sep_len + pl - 1

        pred_logits = logits[i, target_start:target_start + al, :]
        pred_toks = torch.argmax(pred_logits, dim=-1).cpu().numpy()
        gt_toks = batch.ans_toks[i, :al].cpu().numpy()

        print(f'--- Sample {i} ({n_chunks} ctx chunks) ---')
        for ci in range(n_chunks):
            chunk_toks = batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
            print(f'  ctx chunk {ci}: {tkz.decode(chunk_toks, skip_special_tokens=True)[:200]}...')
        chunk_offset += n_chunks

        prompt_ids = batch.prompt_toks[i, :pl].cpu().numpy()
        print(f'  prompt:     {tkz.decode(prompt_ids)}')
        print(f'  GT answer:  {tkz.decode(gt_toks)}')
        print(f'  TF predict: {tkz.decode(pred_toks)}')
        print()

    # --- Autoregressive generation ---
    gen_toks = generate_qna(
        model, batch,
        max_new_tokens=max_new_tokens, temperature=0,
    )
    gen_np = gen_toks.cpu().numpy()

    print(f'=== {split_name} — autoregressive generation (max_new_tokens={max_new_tokens}) ===')
    print(f'gen_toks shape: {gen_toks.shape}')
    print()

    chunk_offset = 0
    for i in range(batch_size):
        n_chunks = batch.ctx_chunk_counts[i]
        pl = batch.prompt_lengths[i]
        al = int(batch.ans_att_mask[i].sum().item())
        gt_toks = batch.ans_toks[i, :al].cpu().numpy()
        prompt_ids = batch.prompt_toks[i, :pl].cpu().numpy()
        gen_ids = gen_np[i]

        print(f'--- Sample {i} ({n_chunks} ctx chunks) ---')
        for ci in range(n_chunks):
            chunk_toks = batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
            print(f'  ctx chunk {ci}: {tkz.decode(chunk_toks, skip_special_tokens=True)[:200]}...')
        chunk_offset += n_chunks

        print(f'  prompt:    {tkz.decode(prompt_ids)}')
        print(f'  GT answer: {tkz.decode(gt_toks)}')
        print(f'  generated: {tkz.decode(gen_ids)}')
        if batch.answerable is not None:
            print(f'  answerable: {batch.answerable[i]}')
        print()

## Inference on train split

In [ ]:
run_qna_inference(model, tkz, qna_agg_train, split_name='TRAIN', batch_off=0, batch_size=5, max_new_tokens=60)

## Inference on validation split

In [ ]:
run_qna_inference(model, tkz, qna_agg_val, split_name='VAL', batch_off=0, batch_size=5, max_new_tokens=60)

## Inference on test split

In [ ]:
run_qna_inference(model, tkz, qna_agg_test, split_name='TEST', batch_off=0, batch_size=5, max_new_tokens=60)

## More examples from different offsets

In [ ]:
run_qna_inference(model, tkz, qna_agg_train, split_name='TRAIN', batch_off=100, batch_size=5, max_new_tokens=60)

In [ ]:
run_qna_inference(model, tkz, qna_agg_val, split_name='VAL', batch_off=100, batch_size=5, max_new_tokens=60)

In [ ]:
run_qna_inference(model, tkz, qna_agg_test, split_name='TEST', batch_off=100, batch_size=5, max_new_tokens=60)